In [35]:
import pathlib

import numpy as np
import pandas as pd

In [36]:
from ete3 import Tree

In [37]:
from splitter import RandomSplitter, LargeTreeTraverseOOCSplitter

## Read phenotype and feature data

Get X and y 

In [38]:
data_folder = pathlib.Path("../../data/processed/biolog/")

In [39]:
from trait_prediction.main import PhenotypeSet, PhenotypeIndex
from trait_prediction.utils import read_generic_features

In [40]:
pheno_file = data_folder / "phenotypes/leaf_phenotypes.tsv"
# feature_file = data_folder / "features_reduced/kofam/kofam_leaf_features_reduced.tsv"
feature_file = data_folder / "features_combined/leaf/Alanine_features_combined.tsv"
pheno_file.is_file(), feature_file.is_file()

(True, True)

In [41]:
phenotype_index = PhenotypeIndex(name="Alanine", category="leaf_biolog")

In [42]:
def generate_data(pheno_file, feature_file, phenotype_index):
    phenotypeset = PhenotypeSet.read_data(pheno_file)
    phenotype = phenotypeset[phenotype_index]
    feature = read_generic_features(
        feature_file, bool_conversion=True, dtype="uint8"
    )
    common_index = list(set(phenotype.phenotype_data.index) & set(feature.index))
    y = phenotype.phenotype_data.loc[common_index]
    X = feature.loc[common_index]
    return X, y, common_index

In [43]:
X, y, common_index = generate_data(pheno_file, feature_file, phenotype_index)

In [44]:
X

,1215_4813,1393_3616,120_6721,1325_2712,1365_1863,992_228,989_480,1363_5044,733_1793,1132_1453,...,UniRef70_UPI00226A1AF4,UniRef70_A0A2U1C6J9,UniRef70_UPI0013A68AD7,UniRef70_A0A1B1BIY0,UniRef70_UPI001E3CF035,UniRef70_A0A097EEC4,UniRef70_A0A3R9AME7,UniRef70_UPI001C87EFB6,UniRef70_A0A3C1DZ99,UniRef70_A0A1E5MF57
genomeID,,,,,,,,,,,,,,,,,,,,,
GCF_001422405.1,0,1,0,0,1,1,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
GCF_001422165.1,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,1,0,0,0,0,0
GCF_001422425.1,0,0,0,0,0,0,1,0,0,1,...,0,0,0,0,1,0,0,0,0,0
GCF_001421605.1,1,0,0,0,0,1,0,0,0,0,...,1,0,0,0,0,0,0,1,0,0
GCF_001421665.1,1,1,0,1,0,0,0,1,0,0,...,1,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GCF_001421535.1,1,1,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
GCF_001423125.1,1,0,0,0,0,1,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
GCF_003258395.1,1,0,0,0,0,1,0,1,1,0,...,0,0,0,0,0,0,1,0,0,0


In [45]:
y

genomeID
GCF_001422405.1    1
GCF_001422165.1    0
GCF_001422425.1    0
GCF_001421605.1    1
GCF_001421665.1    1
                  ..
GCF_001421535.1    1
GCF_001423125.1    1
GCF_003258395.1    1
GCF_001425445.1    0
GCF_001424045.1    1
Name: Alanine--leaf_biolog, Length: 195, dtype: uint8

## Read the tree and create train/test split

Create X_train, X_test, y_train, y_test

In [46]:
tree_file = pathlib.Path("../../data/processed/biolog/phylogeny/v2_all_combined_trimmed_tree-labels.newick")
tree_file.is_file()

True

In [47]:
tree = Tree(str(tree_file), format=1)
tree

Tree node '' (0x7f8a3ae50a9)

In [48]:
# Get the number of leaves in the tree
len(tree)

1361

In [49]:
tree.get_leaf_names()

['Chitinophaga_sp._OK723_2757320416.fna',
 'Chitinophaga_sp._YR573_2690315653.fna',
 'Chitinophaga_sp._CF118_2687453677.fna',
 'Chitinophaga_sp._CF418_2609459644.fna',
 'Chitinophaga_sp._YR627_2693429788.fna',
 'Niastella_sp._CF465_2738541278.fna',
 'Filimonas_sp._YR581_2757320532.fna',
 'FW305-C-49.genome',
 'FW305-C-271.genome',
 'FW305-C-272.genome',
 'FW305-C-21.genome',
 'Hymenobacter_sp._YR204_2739367866.fna',
 '926556.3.RAST',
 'GW458-12-2-14-TSB2.genome',
 'Larkinella_sp._BK230_2802428837.fna',
 'GCF_001424405.1.fna',
 'Dyadobacter_sp._SG02_2609459762.fna',
 'Mucilaginibacter_sp._OK098_2609459642.fna',
 'Mucilaginibacter_sp._OV119_2757320427.fna',
 'Mucilaginibacter_sp._YR332_2616644844.fna',
 'Mucilaginibacter_sp._OK268_2690315638.fna',
 'Mucilaginibacter_sp._OK283_2684623006.fna',
 'Pedobacter_sp._OK701_2738541283.fna',
 'FW305-3-2-15-E-R2A2.genome',
 'Pedobacter_sp._YR016_2738541284.fna',
 'Pedobacter_sp._OK628_2738543023.fna',
 'Pedobacter_sp._OK626_2609459674.fna',
 'GCF_0

In [50]:
def prune_tree(tree, dataset_to_keep, common_index):
    if dataset_to_keep == "at_leaf":
        for leaf in tree:
            if leaf.name.startswith("GCF"):
                leaf.name = leaf.name.rstrip(".fna")
            else:
                leaf.delete()
    elif dataset_to_keep == "literature":
        for leaf in tree:
            if leaf.name.endswith("RAST"):
                leaf.name = leaf.name.rstrip(".RAST")
            else:
                leaf.delete()
    else:
        raise ValueError("dataset_to_keep must be either 'at_leaf' or 'literature'")
    # Are all the leaves in our tree present in common_index?
    is_subset = set([leaf.name for leaf in tree]) <= set(common_index)
    if not is_subset:
        # remove leaves that are not in the common_index
        tree_leaves_not_in_index = set([leaf.name for leaf in tree]) - set(common_index)
        for leaf in tree_leaves_not_in_index:
            tree.search_nodes(name=leaf)[0].delete()

In [51]:
prune_tree(tree, "at_leaf", common_index)

In [52]:
# new number of leaves in tree
len(tree)

175

In [53]:
final_common_index = set([leaf.name for leaf in tree])

### Random split

In [54]:
random_splitter = RandomSplitter(test_set_ratio=0.2)
samples = list(final_common_index)
test_samples_rs = random_splitter.split(samples)
train_samples_rs = list(set(samples) - set(test_samples_rs))

In [55]:
X_train_rs = X.loc[train_samples_rs]
X_test_rs = X.loc[test_samples_rs]
y_train_rs = y.loc[train_samples_rs]
y_test_rs = y.loc[test_samples_rs]
print(X_train_rs.shape, X_test_rs.shape, y_train_rs.shape, y_test_rs.shape)

(140, 1000) (35, 1000) (140,) (35,)


### Out-of-clade split

In [56]:
ooc_splitter = LargeTreeTraverseOOCSplitter(
    tree,
    test_set_range=(0.2, 0.3),
    single_clades=None,
    n_max_clade=2,
    prefer_small_clade=False,
    growth_data=None,
    min_zeros=0,
    min_ones=0,
    time_out_iter=None,
)
samples = list(final_common_index)
test_samples_ooc = ooc_splitter.split(samples)
train_samples_ooc = list(set(samples) - set(test_samples_ooc))

In [57]:
X_train_ooc = X.loc[train_samples_ooc]
X_test_ooc = X.loc[test_samples_ooc]
y_train_ooc = y.loc[train_samples_ooc]
y_test_ooc = y.loc[test_samples_ooc]
print(X_train_ooc.shape, X_test_ooc.shape, y_train_ooc.shape, y_test_ooc.shape)

(134, 1000) (41, 1000) (134,) (41,)


## Is the OOC split choosing different samples each time?

In [58]:
import random
from functools import reduce

def get_clades(single_clades, test_set_range, samples):
    while True:
        clades = random.sample(single_clades, 2)
        test_samples = reduce(np.union1d, clades)
        if (len(test_samples) / len(samples)) < test_set_range[0] or (
            len(test_samples) / len(samples)
        ) > test_set_range[1]:
            continue
        else:
            break
    return test_samples


In [59]:
test_set_range = (0.2, 0.3)
single_clades = ooc_splitter.compute_single_clades(tree, samples)

print(f"Number of single clades = {len(single_clades)}")

clades_1 = get_clades(single_clades, test_set_range, samples)
clades_2 = get_clades(single_clades, test_set_range, samples)
common_elements = set(clades_1) & set(clades_2)
perc_common = len(common_elements) / min(len(clades_1), len(clades_2)) * 100
print(f"Percentage of common elements = {perc_common}")

Number of single clades = 168
Percentage of common elements = 94.44444444444444


Yes, sometimes it does.

## Machine learning performance

In [60]:
from catboost import CatBoostClassifier
def make_classifier(random_state: int, categorical_feature_names: list[str] | None):
    """
    Creates a classifier.

    Parameters
    ---------
    random_state : int
        Random state.
    categorical_feature_names : list[str] | None
        List of categorical feature names.

    Returns
    ------
    Classifier object.
    """
    clf = CatBoostClassifier(
        # iterations=1000,
        # depth=8,
        # learning_rate=0.03,
        # l2_leaf_reg=3,
        # bootstrap_type="Bayesian",
        # bagging_temperature=1,
        random_state=random_state,
        objective="Logloss",
        cat_features=categorical_feature_names,
        verbose=False,
        allow_writing_files=False,
        thread_count=1,
    )
    return clf


In [61]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
)
def get_scores(
    y_true: pd.Series, y_pred: pd.Series, index_name: str
) -> pd.DataFrame:
    """
    Calculates the scores for the given true and predicted labels.

    Parameters
    ---------
    y_true : pd.DataFrame
        True labels.
    y_pred : pd.DataFrame
        Predicted labels.

    Returns
    ------
    pd.DataFrame
        Scores.
    """
    scores = {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "matthews_corrcoef": matthews_corrcoef(y_true, y_pred),
    }
    return pd.DataFrame(scores, index=[index_name])


In [62]:
categorical_feature_names = list(X.columns)
clf_rs = make_classifier(42, categorical_feature_names)
clf_ooc = make_classifier(42, categorical_feature_names)

### Random split performance

In [63]:
clf_rs.fit(X_train_rs, y_train_rs)
y_predict_rs = clf_rs.predict(X_test_rs)

In [64]:
scores_rs = get_scores(y_test_rs, y_predict_rs, "random split")
scores_rs

,accuracy,balanced_accuracy,f1,precision,recall,matthews_corrcoef
random split,0.885714,0.889803,0.882353,0.833333,0.9375,0.777053


random split	0.888889	0.896104	0.904762	0.95	0.863636	0.777212

### OOC split performance

In [65]:
clf_ooc.fit(X_train_ooc, y_train_ooc)
y_predict_ooc = clf_ooc.predict(X_test_ooc)

In [66]:
scores_ooc = get_scores(y_test_ooc, y_predict_ooc, "ooc split")
scores_ooc

,accuracy,balanced_accuracy,f1,precision,recall,matthews_corrcoef
ooc split,0.829268,0.651042,0.461538,0.75,0.333333,0.421398


ooc split	0.761905	0.814815	0.75	0.6	1.0	0.614636

## Features

In [67]:
# Get the top 10 most important features
clf_rs.get_feature_importance(prettified=True)[:10]

,Feature Id,Importances
0,K01271,5.008399
1,K00459,2.425798
2,K01969,1.879603
3,K22210,1.870121
4,K03453.1,1.710666
5,952_631.1,1.610139
6,K05989,1.373288
7,K01692,1.294914
8,K09764,1.244509
9,K01637,1.243259


In [68]:
clf_ooc.get_feature_importance(prettified=True)[:10]

,Feature Id,Importances
0,K01637.1,3.299178
1,K00854,3.010258
2,K01637,2.907888
3,K07481,2.416218
4,K21686.1,2.109997
5,K09764.1,1.863118
6,K09764,1.831932
7,K03453.1,1.320571
8,K19221,1.102433
9,K00459,1.039242
